In [3]:
import pandas as pd

missions = pd.read_excel("data/Données_du_site_volontariat_du_16octobre2026_corrige_v2.xlsx", sheet_name="missions")
missions.head()

,id,titre,structure_accueil,categorie,type_volontariat,lieu,duree_jours,nb_volontaires,indemnite,date_debut,date_fin,date_limite,disponible
0,1,Création de Contenus Numériques de Sensibilisa...,Agence Nationale Pour l'Emploi,Numérique,Volontariat en Ligne,En ligne,21,5,20000,2026-11-11,2026-12-01,2026-10-28,Oui
1,2,Assainissement des Quartiers de Porto-Novo,Agence Nationale Pour l'Emploi,Environnement,Volontariat Communautaire,"Porto-Novo, Ouémé",21,40,3500,2026-11-08,2026-11-28,2026-10-25,Oui
2,3,Sensibilisation à la Santé Maternelle,ONG Santé Communautaire Bénin,Santé,Volontariat International,Ouagadougou,365,4,120000,2026-11-14,2027-11-13,2026-10-31,Oui
3,4,Coordination de Projets d'Accès à l'Eau,ONG Santé Communautaire Bénin,Développement,Volontariat de Solidarité Internationale (VSI),Niamey,365,3,150000,2026-11-24,2027-11-23,2026-11-10,Oui
4,5,Assistant Chef de Projet Export,Agence Nationale Pour l'Emploi,Développement,Volontariat International en Entreprise (VIE),Lyon,365,2,450000,2026-12-01,2027-11-30,2026-11-17,Oui


In [4]:
import pandas as pd

competences = pd.read_excel("data/Données_du_site_volontariat_du_16octobre2026_corrige_v2.xlsx", sheet_name="competences")
mc=pd.read_excel("data/Données_du_site_volontariat_du_16octobre2026_corrige_v2.xlsx", sheet_name="missions_competences")
print(competences.shape)
print(mc.shape)

(15, 2)
(34, 2)


In [5]:
import pandas as pd

missions = pd.read_excel("data/Données_du_site_volontariat_du_16octobre2026_corrige_v2.xlsx", sheet_name="missions")
missions.dtypes

id                            int64
titre                           str
structure_accueil               str
categorie                       str
type_volontariat                str
lieu                            str
duree_jours                   int64
nb_volontaires                int64
indemnite                     int64
date_debut           datetime64[us]
date_fin             datetime64[us]
date_limite          datetime64[us]
disponible                      str
dtype: object

In [6]:
import pandas as pd

aujourdhui = pd.Timestamp.today().normalize()
print(aujourdhui)
condition = missions["date_limite"] >= aujourdhui
print(condition)
disponibles = missions[condition]
print(disponibles)

2026-10-29 00:00:00
0     False
1     False
2      True
3      True
4      True
5      True
6     False
7     False
8     False
9      True
10     True
11     True
Name: date_limite, dtype: bool
    id                                      titre  \
2    3      Sensibilisation à la Santé Maternelle   
3    4    Coordination de Projets d'Accès à l'Eau   
4    5            Assistant Chef de Projet Export   
5    6  Appui aux Centres de Santé Communautaires   
9   10          Campagne Nationale de Reboisement   
10  11        Appui Scolaire aux Écoles Primaires   
11  12    Formation aux Premiers Secours Civiques   

                  structure_accueil      categorie  \
2     ONG Santé Communautaire Bénin          Santé   
3     ONG Santé Communautaire Bénin  Développement   
4    Agence Nationale Pour l'Emploi  Développement   
5   Association Éducation Pour Tous          Santé   
9            Coopérative Agro-Verte  Environnement   
10  Association Éducation Pour Tous      Éducation   
11

In [7]:
mc=pd.read_excel("data/Données_du_site_volontariat_du_16octobre2026_corrige_v2.xlsx", sheet_name="missions_competences")
competences_par_mission = mc.groupby("mission_id")["competence"].apply(set)
print(competences_par_mission)

mission_id
1                           {Créativité, Communication}
2     {Endurance physique, Sensibilisation environne...
3     {Premiers secours, Patience et écoute, Communi...
4     {Observation de terrain, Collecte de données, ...
5                  {Collecte de données, Communication}
6     {Premiers secours, Patience et écoute, Communi...
7     {Animation de groupe, Créativité, Arts plastiq...
8     {Sensibilisation environnementale, Travail en ...
9     {Connaissance de la faune, Collecte de données...
10    {Endurance physique, Sensibilisation environne...
11    {Français et mathématiques, Patience et écoute...
12    {Prise de parole en public, Premiers secours, ...
Name: competence, dtype: object


In [8]:
volontaire = {"Communication", "Travail en équipe", "Collecte de données",
              "Premiers secours", "Patience et écoute"}
de_la_mission = competences_par_mission[3]
en_commun = volontaire & de_la_mission
score = len(en_commun) / len(de_la_mission)
print(score)

1.0


In [9]:
def calculer_score(competences_volontaire, competences_mission):
    if len(competences_mission) == 0:
        return 0.0
    en_commun = competences_volontaire & competences_mission
    score = len(en_commun) / len(competences_mission)
    return score
print(calculer_score(volontaire, competences_par_mission[3]))  # 1.0
print(calculer_score(volontaire, competences_par_mission[5]))  # 1.0
print(calculer_score(volontaire, competences_par_mission[4]))  # 0.666...
print(calculer_score(volontaire, competences_par_mission[1]))  # 0.5
print(calculer_score(volontaire, competences_par_mission[2]))  # 0.333...
print(calculer_score(set(), competences_par_mission[3]))       # 0.0
print(calculer_score(volontaire, set()))                       # 0.0, sans erreur

1.0
1.0
0.6666666666666666
0.5
0.3333333333333333
0.0
0.0


In [10]:
disponibles = disponibles.copy()
disponibles["score"] = disponibles["id"].apply(
    lambda mission_id: calculer_score(volontaire, competences_par_mission[mission_id])
)
disponibles[["id", "titre", "score"]]

,id,titre,score
2,3,Sensibilisation à la Santé Maternelle,1.000000
3,4,Coordination de Projets d'Accès à l'Eau,0.666667
4,5,Assistant Chef de Projet Export,1.000000
5,6,Appui aux Centres de Santé Communautaires,1.000000
9,10,Campagne Nationale de Reboisement,0.333333
10,11,Appui Scolaire aux Écoles Primaires,0.333333
11,12,Formation aux Premiers Secours Civiques,0.333333


In [11]:
SEUIL_LONGUE_DUREE = 60

def est_compatible(duree_jours, disponibilite):
    if duree_jours <= SEUIL_LONGUE_DUREE:
        categorie = "courte"
    else:
        categorie = "longue"
    return categorie == disponibilite

In [12]:
def recommander(competences, disponibilite, aujourdhui=None):
    if aujourdhui is None:
        aujourdhui = pd.Timestamp.today().normalize()

    # 1. missions disponibles
    res = missions[missions["date_limite"] >= aujourdhui].copy()

    # 2. volontaire sans compétence : tri par date limite
    if len(competences) == 0:
        res["score"] = 0.0
        res = res.sort_values(by=["date_limite", "id"])
    else:
        # 3. score, compétences en commun, compatibilité de durée
        res["score"] = res["id"].apply(
            lambda i: calculer_score(competences, competences_par_mission.get(i, set()))
        )
        res["a_competences"] = res["score"] > 0
        res["compatible"] = res["duree_jours"].apply(
            lambda d: est_compatible(d, disponibilite)
        )
        # 4. tri en 5 critères
        res = res.sort_values(
            by=["a_competences", "compatible", "score", "date_limite", "id"],
            ascending=[False, False, False, True, True],
        )

    return res[["id", "titre", "score", "duree_jours", "date_limite"]]

In [13]:
d = pd.Timestamp("2026-10-23")

# Test 1 : sans compétence
print(recommander(set(), "courte", d)["id"].tolist())

# Test 2 : longue durée, 3 compétences
v = {"Premiers secours", "Patience et écoute", "Communication"}
print(recommander(v, "longue", d)["id"].tolist())

# Test 3 : aucune compétence en commun, courte durée
print(recommander({"Arts plastiques"}, "courte", d)["id"].tolist())

[2, 9, 1, 3, 6, 10, 4, 11, 5, 12]
[3, 5, 6, 1, 11, 12, 4, 2, 9, 10]
[2, 9, 1, 6, 10, 11, 12, 3, 4, 5]
